# Custom Surveys

This notebook demonstrates how to support a new survey by creating a custom `ObsTable` subclass. We discuss what information is needed by the survey, which functions must be created, and how use the new survey.

## Survey Data

`LightCurveLynx` requires a few different pieces of information from the survey:

  * where the survey is looking (time, RA, dec, fov)
  * the noise characteristics of the observations (zero point, magnitude limit)
  * the information about the filters and their transmission curves

This data can come from a simulation or a real survey.

This data is inside the `ObsTable` object in one of two internal data structures:

  * `_table` is a Pandas data frame that holds all of the per-observation data. At a minimum the table requires columns for time (in MJD), RA (in degrees), and dec (in degrees). Many surveys include a lot of other data in this table, such as the filter used for the observation and the seeing at the time of the observation.
  * `survey_values` is a dictionary of constants for the survey. It can include information such as dark current for the detectors or radius of the observation (in degrees).

### Example

Let's start by looking at one of the fake data sets included in the test data. This data is in the Rubin OpSim format, so we load it with the `OpSim` subclass of `ObsTable`.

In [ ]:
from lightcurvelynx.obstable.opsim import OpSim

opsim = OpSim.from_db("../../tests/lightcurvelynx/data/opsim_small.db")
print(f"Loaded OpSim with {len(opsim)} rows.")

print("\nSurvey Constants:")
for key, value in opsim.survey_values.items():
    print(f"  {key}: {value}")

print("\nFirst 5 rows of the OpSim table:")
opsim._table.head()

Data from either the survey values or the survey table can be accessed using the `[]` notation.

In [ ]:
# Access a constant value for the survey.
print("ccd_pixel_width: ", opsim["ccd_pixel_width"])

# Access the table values for a specific column.
print("\nRA values: ", opsim["ra"])

### Pointing Information

As noted above, the survey's pointing information must be provided by three columns with predefined names `time`, `ra`, and `dec` with units of days, degrees, and degrees respectively. This information, along with either the fov's radius or the detector footprint, are used to determine the times at which a give object is observed.

We can also use it to estimate coverage of the survey or plot the overall footprint.

In [ ]:
opsim.plot_footprint()

## Creating a New Survey Type

There are only a few functions and parameters that need to be set to create a new `ObsTable` subclass. We first describe these below and then present a code block for an example subclass.

### Constructor

At a minimum, the `__init__` function needs to take the Pandas table of input values and pass it along to the parent class's constructor. In addition, it can do whatever other setup or preprocessing is needed.

You can also provide it a column map, which automatically renames the columns to match what is expected by the code. Each key provides the final column name and its value can either be a string or list of strings of possible names in the data. For example if the column map is `{"ra": ["fieldRA", "osbRA"]}`, the code will check the input table either "fieldRA" or "obsRA" and rename that column to "ra". If there are multiple matches, only the first will be used.

### Survey Values

The `ObsTable` class uses a `_default_survey_values` class attribute to provide the default survey constants, such as "dark_current", "pixel_scale", or "radius". The subclass can define its own `_default_survey_values` class attribute with its corresponding values.

The only survey value that is required if the radius of the camera's field of view (`radius`).

We highly recommend including the units in a comment after each parameter.

### Noise Information

The information about the observation's noise characteristics varies per-survey, so the `ObsTable` class relies one two helper functions that can be implemented by a custom survey:

  * `_assign_zero_points(self)` - Computes the zero points for each row in nJy (which produces 1 e-) and appends them as an column called `zp`. If the input table already includes zero point information (in the column named `zp`) this function can be skipped.
  * `bandflux_error_point_source(self, bandflux, index)` - Computes the observational bandflux error for a given source from the provided bandflux (in nJy). It only works on the subset of observations corresponding to the given index array. The output is used by the simulation's `apply_noise()` function.

In [ ]:
from lightcurvelynx.astro_utils.mag_flux import mag2flux
from lightcurvelynx.astro_utils.noise_model import poisson_bandflux_std
from lightcurvelynx.obstable.obs_table import ObsTable


class ExampleObsTable(ObsTable):
    """An example ObsTable subclass."""

    # Default survey values (using numbers for LSSTCam).
    _default_survey_values = {
        "dark_current": 0.022,  # electrons per second per pixel
        "gain": 1.595,  # electrons per ADU
        "pixel_scale": 0.2,  # arcsec/pixel
        "radius": 1.75,  # degrees
        "read_noise": 5.82,  # readout electrons per pixel
        "survey_name": "my_awesome_survey",
    }

    _default_colmap = {
        "filter": "band",  # Map band->filter
        "zp": "zeroPoint",  # Map zeroPoint->zp
    }

    # The constructor
    def __init__(self, table, colmap=None, **kwargs):
        colmap = self._default_colmap if colmap is None else colmap
        super().__init__(table, colmap=colmap, **kwargs)

    def _assign_zero_points(self):
        """Compute the zero points (in nJy) for each observation using whatever data
        your custom survey has."""
        if "zp" in self._table.columns:
            return  # Nothing to do

        # Let's assume our custom survey provides the zero points as magnitudes as "zp_mag_e"
        # for the electron-based zero point (as a magnitude). We convert it to nJy and add it
        # back as a column.
        if "zp_mag_e" in self._table.columns:
            zp_values = mag2flux(self._table["zp_mag_e"])
            self.add_column("zp", zp_values, overwrite=True)
            return

        raise ValueError("Not enough information to compute the zero points.")

    def bandflux_error_point_source(self, bandflux, index):
        """Compute observational bandflux error for a point source using
        whatever data your survey has on hand.

        Parameters
        ----------
        bandflux : array_like of float
            Band bandflux of the point source in nJy.
        index : array_like of int
            The index of the observation in the LSSTObsTable table.

        Returns
        -------
        flux_err : array_like of float
            Simulated bandflux noise in nJy.
        """
        # Extract the subset of observations corresponding to the given list of indices.
        observations = self._table.iloc[index]

        # For simplicity, we just use the built-in Poisson noise model with some constants,
        # but users can implement their own to reflect the survey's characteristics.
        return poisson_bandflux_std(
            bandflux,
            total_exposure_time=30.0,  # Constant exposure time.
            exposure_count=1,  # Single exposure.
            psf_footprint=10.0,  # PSF footprint in arcsec^2.
            sky=observations["sky_bg_e"],  # sky background in e-
            zp=observations["zp"],  # zero point in nJy
            readout_noise=self.survey_values["read_noise"],
            dark_current=self.survey_values["dark_current"],
            zp_err_mag=0.01,  # Use a constant noise floor for example.
        )

We can create a fake ObsTable from given values. As you can see from the displayed table, the columns have been renamed according to the column map (band->filter) and the zero point column has automatically been added.

In [ ]:
import pandas as pd

table_data = {
    "time": [60676.0, 60676.1, 60676.2],
    "ra": [10.0, 20.0, 10.0],
    "dec": [-10.0, -20.0, -10.0],
    "band": ["r", "r", "r"],
    "zp_mag_e": [25.0, 25.5, 26.0],  # Zero point magnitudes for the example.
    "sky_bg_e": [100.0, 150.0, 200.0],  # Sky background in electrons for the example.
}
pdf = pd.DataFrame(table_data)

obs_table = ExampleObsTable(pdf)
obs_table.head()

## Using Tables in Simulations

The new `ObsTable` can be used in a simulation like any other table. To complete the simulation, we also need a model and the transmission curves for the survey's passband.

We start be loading the passbands that we will use. In this case we use the passbands from the LSST preset (but use a cached local version in the test directory to avoid a download). In most cases users will want to use `data/passbands/` from the root directory.

In [ ]:
from lightcurvelynx.astro_utils.passbands import PassbandGroup

table_dir = "../../tests/lightcurvelynx/data/passbands"
passband_group = PassbandGroup.from_preset(preset="LSST", table_dir=table_dir)

Then we define a simple sinwave model for the simulation.  We choose the position (RA, dec) to match two of the three observations in our fake table.

In [ ]:
from lightcurvelynx.models.basic_models import SinWaveModel

source = SinWaveModel(
    brightness=1e16,
    amplitude=2e15,
    frequency=11.0,
    ra=10.0,
    dec=-10.0,
    t0=0.0,
    node_label="sin_wave_source",
)

Then we run the simulation. 

In [ ]:
from lightcurvelynx.simulate import simulate_lightcurves

lightcurves = simulate_lightcurves(source, 1, obs_table, passband_group)
lightcurves.iloc[0]["lightcurve"]

As we can see from the lightcurve table, the object is observed at the correct two times (MJD 60676.0 and 60676.2) with different pre-noise fluxes (`flux_perfect`) corresponding to different points on the sin wave. Each observation also has added noise.

## Warnings and Caveats

The most common problems we have encountered when creating a new survey type are understanding the survey's columns and ensuring the correct units. For example consider a column named `sky_bg` that holds the sky background. Even given this knowledge, the user working this column needs to know if the background is given in electron, ADU, or something else. We recommend thoroughly validating the output produced by each new survey to ensure accuracy.